# Cheat Sheet — Laptop Price Prediction (Extended Project)

Quick-reference syntax for every technique used in this project. Snippets use small dummy data so every cell runs standalone — copy the *pattern*, not the literal variable names, into your working notebook.

Sections: Setup · Inspection · String Cleaning · Feature Engineering · EDA & Stats · Encoding · Splitting · Models · Tuning & CV · Evaluation · Feature Importance · Persistence.


In [ ]:
# Run this first so every snippet below works standalone
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42


## 1. Loading & Inspection

In [ ]:
# Load
df = pd.read_csv('laptop_price.csv')

# Quick look
df.head()          # first 5 rows
df.shape            # (rows, cols)
df.info()           # dtypes + non-null counts
df.describe().T     # numeric summary stats, transposed for readability
df.describe(include='object').T  # categorical summary (count, unique, top, freq)
df.columns.tolist() # list of column names
df.dtypes           # dtype per column


## 2. Data Quality Audit

In [ ]:
# Dtype audit table pattern
audit = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'n_unique': df.nunique(),
    'n_missing': df.isnull().sum(),
    'sample': [df[c].dropna().unique()[:3].tolist() for c in df.columns]
})
audit

# Duplicates
df.duplicated().sum()
df.drop_duplicates(inplace=True)   # if needed

# Unique values per object column (scan for disguised missing / unit text)
for c in df.select_dtypes(include='object').columns:
    print(c, '->', df[c].unique()[:10])


## 3. String Cleaning (unit-bearing text -> numeric)

In [ ]:
# Pattern: strip a suffix, then cast
example = pd.Series(['4 GB', '8 GB', '16 GB'])
example.str.replace(' GB', '').astype(int)

# Idempotent version (safe to re-run): cast to str FIRST
s = pd.Series([64, '64-bit'])   # mixed clean/unclean
s.astype(str).str.replace('-bit', '').astype(int)

# Chained replace for two variants ("stars" and "star")
ratings = pd.Series(['3 stars', '1 star'])
ratings.astype(str).str.replace(' stars', '').str.replace(' star', '').astype(int)

# Replace a sentinel BEFORE stripping a suffix
gen = pd.Series(['10th', 'Not Available'])
gen.astype(str).str.replace('Not Available', '0').str.replace('th', '').astype(int)

# .str.extract for pulling digits out with regex (alternative approach)
pd.Series(['4 GB', '16 GB']).str.extract(r'(\d+)').astype(int)


## 4. Feature Engineering

In [ ]:
# Loop-based cleaning across several similarly-shaped columns
cols_to_clean = ['ram_gb', 'ssd', 'hdd', 'graphic_card_gb']
for col in cols_to_clean:
    df[col] = df[col].astype(str).str.replace(' GB', '').astype(int)

# Composite / arithmetic features
df['total_storage'] = df['ssd'] + df['hdd']
df['has_ssd'] = (df['ssd'] > 0).astype(int)
df['has_hdd'] = (df['hdd'] > 0).astype(int)

# Safe ratio (avoid divide-by-zero)
df['ssd_ratio'] = np.where(df['total_storage'] > 0, df['ssd'] / df['total_storage'], 0)

# Ordinal mapping for a domain-known hierarchy
tier_map = {
    'Celeron Dual': 0, 'Pentium Quad': 0,
    'Core i3': 1, 'Ryzen 3': 1,
    'Core i5': 2, 'Ryzen 5': 2,
    'Core i7': 3, 'Ryzen 7': 3,
    'Core i9': 4, 'Ryzen 9': 4, 'M1': 4,
}
df['processor_tier'] = df['processor_name'].map(tier_map)

# Boolean flag from a category
df['is_gaming'] = (df['weight'] == 'Gaming').astype(int)

# Log-transform a skewed count feature
df['popularity_score'] = np.log1p(df['Number of Ratings'] + df['Number of Reviews'])

# Drop columns once replaced by engineered features
df.drop(columns=['ssd', 'hdd'], inplace=True)


## 5. EDA & Statistics

In [ ]:
# Skewness of the target
df['Price'].skew()

# Log-transform target for modeling a right-skewed variable
df['log_price'] = np.log1p(df['Price'])

# Correlation heatmap
numeric_df = df.select_dtypes(include='number')
plt.figure(figsize=(12, 8))
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.show()

# Variance Inflation Factor (multicollinearity)
from statsmodels.stats.outliers_influence import variance_inflation_factor
X_vif = numeric_df.drop(columns=['Price']).dropna()
vif = pd.DataFrame({
    'feature': X_vif.columns,
    'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
}).sort_values('VIF', ascending=False)
vif

# Boxplot: categorical/ordinal feature vs target
sns.boxplot(data=df, x='ram_gb', y='Price', order=sorted(df['ram_gb'].unique()))
plt.show()

# IQR-based outlier detection
Q1, Q3 = df['Price'].quantile([0.25, 0.75])
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
outliers = df[(df['Price'] < lower) | (df['Price'] > upper)]
len(outliers)


## 6. Encoding Categorical Variables

In [ ]:
# Bucket columns by cardinality
cat_cols = df.select_dtypes(include='object').columns
low_card = [c for c in cat_cols if df[c].nunique() < 5]
high_card = [c for c in cat_cols if df[c].nunique() >= 5]

# One-hot encoding (low cardinality)
df = pd.get_dummies(df, columns=low_card, drop_first=True)

# --- Leakage-safe target encoding (high cardinality) ---
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Price'])
y = df['Price']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

for col in high_card:
    # 1. Learn the mapping from TRAIN ONLY
    means = y_train.groupby(X_train[col]).mean()
    global_mean = y_train.mean()
    # 2. Apply to both splits
    X_train[col] = X_train[col].map(means)
    X_test[col] = X_test[col].map(means).fillna(global_mean)  # unseen category -> global mean

# Simpler (leaky) version — for reference only, avoid in real evaluation:
# df[col] = df[col].map(df.groupby(col)['Price'].mean())   # computed on FULL df -> leakage


## 7. Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

# Train/validation/test (three-way) when you need a tuning set separate from the final test set
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.25, random_state=RANDOM_STATE)


## 8. Models

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Baseline: always predict the mean
y_pred_baseline = np.full_like(y_test, fill_value=y_train.mean(), dtype=float)

lin = LinearRegression().fit(X_train, y_train)
ridge = Ridge(alpha=1.0, random_state=RANDOM_STATE).fit(X_train, y_train)
lasso = Lasso(alpha=1.0, random_state=RANDOM_STATE).fit(X_train, y_train)

rf = RandomForestRegressor(random_state=RANDOM_STATE).fit(X_train, y_train)
gb = GradientBoostingRegressor(random_state=RANDOM_STATE).fit(X_train, y_train)

# Optional: XGBoost, if installed
# from xgboost import XGBRegressor
# xgb = XGBRegressor(random_state=RANDOM_STATE).fit(X_train, y_train)

y_pred = rf.predict(X_test)


## 9. Hyperparameter Tuning & Cross-Validation

In [ ]:
from sklearn.model_selection import cross_val_score, KFold, RandomizedSearchCV, GridSearchCV

# k-fold cross-validation score (note: sklearn maximizes, so error metrics are negated)
scores = cross_val_score(rf, X_train, y_train, cv=5, scoring='neg_mean_absolute_error')
mae_cv = -scores
print(mae_cv.mean(), mae_cv.std())

# RandomizedSearchCV (efficient for large hyperparameter spaces)
param_dist = {
    'n_estimators': [100, 200, 400, 600],
    'max_depth': [None, 5, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2', None],
}
search = RandomizedSearchCV(
    RandomForestRegressor(random_state=RANDOM_STATE),
    param_distributions=param_dist,
    n_iter=25, cv=5, scoring='neg_mean_absolute_error',
    random_state=RANDOM_STATE, n_jobs=-1
)
search.fit(X_train, y_train)
search.best_params_, -search.best_score_

# GridSearchCV (exhaustive, good for a small final grid)
param_grid = {'n_estimators': [200, 400], 'max_depth': [10, 20, None]}
grid = GridSearchCV(RandomForestRegressor(random_state=RANDOM_STATE), param_grid, cv=5,
                     scoring='neg_mean_absolute_error', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = search.best_estimator_


## 10. Evaluation Metrics & Diagnostic Plots

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)   # or np.sqrt(mean_squared_error(...))
r2 = r2_score(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

print(f"MAE: {mae:.2f}  RMSE: {rmse:.2f}  R2: {r2:.3f}  MAPE: {mape:.2f}%")

# Predicted vs actual
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.plot(lims, lims, 'r--')
plt.xlabel('Actual Price'); plt.ylabel('Predicted Price')
plt.show()

# Residual plot
residuals = y_test - y_pred
plt.figure(figsize=(6, 4))
plt.scatter(y_pred, residuals, alpha=0.5)
plt.axhline(0, color='r', linestyle='--')
plt.xlabel('Predicted Price'); plt.ylabel('Residual')
plt.show()

# Residual distribution
sns.histplot(residuals, kde=True)
plt.show()


## 11. Feature Importance

In [ ]:
# Impurity-based (fast, biased toward high-cardinality features)
importances = best_model.feature_importances_
top_idx = np.argsort(importances)[-10:][::-1]
top_features = X_train.columns[top_idx]

plt.figure(figsize=(8, 5))
sns.barplot(x=importances[top_idx], y=top_features)
plt.title('Impurity-based importance (top 10)')
plt.show()

# Permutation importance (slower, measures real predictive damage on held-out data)
from sklearn.inspection import permutation_importance
result = permutation_importance(best_model, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
perm_sorted_idx = result.importances_mean.argsort()[-10:][::-1]

plt.figure(figsize=(8, 5))
sns.barplot(x=result.importances_mean[perm_sorted_idx], y=X_test.columns[perm_sorted_idx])
plt.title('Permutation importance (top 10)')
plt.show()


## 12. Persistence & Inference

In [ ]:
import joblib

# Save
joblib.dump(best_model, 'laptop_price_model.pkl')
joblib.dump({'target_encoding_maps': {}, 'one_hot_columns': list(X_train.columns)}, 'laptop_price_encoders.pkl')

# Load
loaded_model = joblib.load('laptop_price_model.pkl')

# Minimal inference wrapper pattern
def predict_price(raw_dict, model, encoders):
    # 1. put raw_dict into a one-row DataFrame
    # 2. re-apply the SAME cleaning / feature engineering / encoding steps used in training
    # 3. reindex to the training column order (X_train.columns), filling missing dummy cols with 0
    # 4. return model.predict(row)[0]
    pass


## Quick lookup: metric formulas

| Metric | Formula | Notes |
|---|---|---|
| MAE | mean(&#124;y - ŷ&#124;) | same units as target, robust to outliers |
| RMSE | sqrt(mean((y - ŷ)²)) | penalizes large errors more than MAE |
| R² | 1 - SS_res / SS_tot | proportion of variance explained, can be negative |
| MAPE | mean(&#124;(y - ŷ) / y&#124;) × 100 | scale-free %, undefined/unstable near y=0 |

## Quick lookup: when to one-hot vs. target-encode

| Cardinality | Encoding | Risk |
|---|---|---|
| < 5 unique values | One-hot (`drop_first=True`) | Minimal — low dimensionality added |
| ≥ 5 unique values | Target/mean encoding | Leakage if fit on full data instead of train only |
